# EE 446 Homework 1 Programming Notebook

Use the **tinyml-arduino** Python environment that you set up for this class. In JupyterLab, select the kernel named **Python (tinyml-arduino)** before running this notebook.

Do not install or uninstall TensorFlow packages inside this notebook. The class environment already contains the required packages for this assignment, including TensorFlow, TensorFlow Model Optimization Toolkit, scikit-learn, NumPy, pandas, and JupyterLab.

This notebook contains the programming questions marked **[Pro]**. Complete each section by replacing the placeholder comments with your own code. Print the requested outputs so that your work can be graded directly from the notebook.


In [61]:
import sys
print(sys.executable)

/Users/asafiron-jobes/ai/projects/tinyml-arduino/bin/python


In [62]:
import sys
!{sys.executable} -m pip install "tensorflow-model-optimization==0.8.0"


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [63]:
import sys
!{sys.executable} -m pip install "keras==2.14.0"


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [64]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix, r2_score

import tensorflow as tf
import tensorflow_model_optimization as tfmot

Sequential = tf.keras.Sequential
Dense = tf.keras.layers.Dense
LSTM = tf.keras.layers.LSTM
to_categorical = tf.keras.utils.to_categorical

print("TensorFlow version:", tf.__version__)
print("TF-MOT version:", tfmot.__version__)

TensorFlow version: 2.14.1
TF-MOT version: 0.8.0



---

# Problem 1: DNN and Wine Classification (80 points)

This problem uses the Wine dataset available through scikit-learn. The dataset is loaded locally from the installed package, so no external data file is required.


In [65]:
# Load the Wine dataset from scikit-learn.
# This avoids requiring an external wine.data file.

wine = load_wine(as_frame=True)

feature_names = list(wine.feature_names)
df = wine.frame.copy()
df["Class"] = wine.target

# Reorder the columns so that the class label appears first.
df = df[["Class"] + feature_names]

# Number of classes
num_classes = df["Class"].nunique()
print("Number of classes:", num_classes)

# Number of features, excluding the class label
num_features = df.shape[1] - 1
print("Number of features:", num_features)

# Basic feature statistics
feature_stats = df.drop(columns=["Class"]).describe().T[["min", "max", "mean", "std"]]
print("\nFeature statistics:\n", feature_stats)

# Class distribution
class_counts = df["Class"].value_counts().sort_index()
print("\nClass distribution:\n", class_counts)


Number of classes: 3
Number of features: 13

Feature statistics:
                                  min      max        mean         std
alcohol                        11.03    14.83   13.000618    0.811827
malic_acid                      0.74     5.80    2.336348    1.117146
ash                             1.36     3.23    2.366517    0.274344
alcalinity_of_ash              10.60    30.00   19.494944    3.339564
magnesium                      70.00   162.00   99.741573   14.282484
total_phenols                   0.98     3.88    2.295112    0.625851
flavanoids                      0.34     5.08    2.029270    0.998859
nonflavanoid_phenols            0.13     0.66    0.361854    0.124453
proanthocyanins                 0.41     3.58    1.590899    0.572359
color_intensity                 1.28    13.00    5.058090    2.318286
hue                             0.48     1.71    0.957449    0.228572
od280/od315_of_diluted_wines    1.27     4.00    2.611685    0.709990
proline                 

## Problem 1 - Part (a)
### Base Model Training and Evaluation


In [66]:
# Step 1: Separate the feature matrix and class labels.
# - Assign the feature columns to variable X.
# - Assign the class labels to variable y.
# - The labels in this scikit-learn dataset are already zero-based: 0, 1, and 2.

# <-- Enter your code here <--#
X = df.drop(columns=["Class"]).values
y = df["Class"].values


In [67]:
# Step 2: Perform a train-test split (70% train, 30% test) using random_state=42

# <-- Enter your code here <--#
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [68]:
# Step 3: Use StandardScaler to normalize the features
# - Fit on X_train and transform both X_train and X_test

# <-- Enter your code here <--#
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [69]:
# Step 4: Use one-hot encoding for y_train and y_test.
# - Use tf.keras.utils.to_categorical.
# - Use num_classes=num_classes to make the output shape explicit.

# <-- Enter your code here <--#
y_train_cat = to_categorical(y_train, num_classes=num_classes)
y_test_cat = to_categorical(y_test, num_classes=num_classes)

In [70]:
# Step 5: Define a Sequential model with the following architecture:
# - Dense(64, activation='relu')
# - Dense(32, activation='relu')
# - Dense(num_classes, activation='softmax')
# Make sure the first Dense layer receives the correct input shape.

# <-- Enter your code here <--#
model = Sequential([
    Dense(64, activation='relu', input_shape=(num_features,)),
    Dense(32, activation='relu'),
    Dense(num_classes, activation='softmax')
])
model.summary()

Model: "sequential_4"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_12 (Dense)            (None, 64)                896       
                                                                 
 dense_13 (Dense)            (None, 32)                2080      
                                                                 
 dense_14 (Dense)            (None, 3)                 99        
                                                                 
Total params: 3075 (12.01 KB)
Trainable params: 3075 (12.01 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [71]:
# Step 6: Compile using Adam optimizer, categorical_crossentropy loss, and accuracy metric
# - Train for 20 epochs with batch_size=8 and validation_split=0.2

# <-- Enter your code here <--#
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
history = model.fit(X_train_scaled, y_train_cat, epochs=20, batch_size=8, validation_split=0.2)

Epoch 1/20
13/13 [==============================] - 0s 6ms/step - loss: 0.9739 - accuracy: 0.4343 - val_loss: 0.9177 - val_accuracy: 0.5200
Epoch 2/20
13/13 [==============================] - 0s 2ms/step - loss: 0.7250 - accuracy: 0.8586 - val_loss: 0.7414 - val_accuracy: 0.6800
Epoch 3/20
13/13 [==============================] - 0s 1ms/step - loss: 0.5566 - accuracy: 0.9596 - val_loss: 0.6121 - val_accuracy: 0.8800
Epoch 4/20
13/13 [==============================] - 0s 1ms/step - loss: 0.4298 - accuracy: 0.9697 - val_loss: 0.5026 - val_accuracy: 0.8800
Epoch 5/20
13/13 [==============================] - 0s 2ms/step - loss: 0.3287 - accuracy: 0.9899 - val_loss: 0.3950 - val_accuracy: 0.8800
Epoch 6/20
13/13 [==============================] - 0s 1ms/step - loss: 0.2494 - accuracy: 0.9899 - val_loss: 0.3141 - val_accuracy: 0.9200
Epoch 7/20
13/13 [==============================] - 0s 1ms/step - loss: 0.1908 - accuracy: 0.9899 - val_loss: 0.2638 - val_accuracy: 0.9200
Epoch 8/20
13/13 [==

In [72]:
# Step 7: Evaluate the model on test data and print:
# - Accuracy
# - Classification report
# - Confusion matrix

# <-- Enter your code here <--#
train_loss, train_acc = model.evaluate(X_train_scaled, y_train_cat, verbose=0)
test_loss, test_acc = model.evaluate(X_test_scaled, y_test_cat, verbose=0)
print(f"Training Accuracy: {train_acc:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

y_pred = np.argmax(model.predict(X_test_scaled), axis=1)
y_true = np.argmax(y_test_cat, axis=1)
print(classification_report(y_true, y_pred))
print(confusion_matrix(y_true, y_pred))

Training Accuracy: 0.9919
Test Accuracy: 1.0000
2/2 [==============================] - 0s 841us/step
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        19
           1       1.00      1.00      1.00        21
           2       1.00      1.00      1.00        14

    accuracy                           1.00        54
   macro avg       1.00      1.00      1.00        54
weighted avg       1.00      1.00      1.00        54

[[19  0  0]
 [ 0 21  0]
 [ 0  0 14]]


In [73]:
# Step 8: Convert the trained model to TFLite format and save it as "model_base.tflite"
# - Print the file size in kilobytes

# <-- Enter your code here <--#
import os

def file_size_kb(path):
    return os.path.getsize(path) / 1024

converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()
with open("model_base.tflite", "wb") as f:
    f.write(tflite_model)
print(f"Base model size: {file_size_kb('model_base.tflite'):.2f} KB")

INFO:tensorflow:Assets written to: /var/folders/1_/r4slzwz964766f02m55dkrzr0000gn/T/tmp0usaim12/assets


INFO:tensorflow:Assets written to: /var/folders/1_/r4slzwz964766f02m55dkrzr0000gn/T/tmp0usaim12/assets


Base model size: 14.14 KB


2026-05-21 11:42:06.040001: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-21 11:42:06.040013: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-21 11:42:06.040123: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/1_/r4slzwz964766f02m55dkrzr0000gn/T/tmp0usaim12
2026-05-21 11:42:06.040653: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-21 11:42:06.040657: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/1_/r4slzwz964766f02m55dkrzr0000gn/T/tmp0usaim12
2026-05-21 11:42:06.042005: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-21 11:42:06.062121: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /var/folders/1_/r4slzwz964766f02m55dkrzr0000gn/T/tmp0usaim12
2026-05-

## Problem 1 - Part (b)

### Quantization (int8, float16, dynamic range)


In [74]:
def representative_data_gen(X_reference, num_samples=100):
    """Create a representative dataset generator for full integer quantization."""
    max_samples = min(num_samples, len(X_reference))
    for i in range(max_samples):
        yield [X_reference[i:i + 1].astype(np.float32)]


def quantize_and_evaluate(model, X_test, y_test_cat, quant_type, filename):
    """Convert a Keras model to TFLite, evaluate it, and report model size.

    Parameters
    ----------
    model : tf.keras.Model
        Trained Keras model.
    X_test : np.ndarray
        Test features after the same preprocessing used for training.
    y_test_cat : np.ndarray
        One-hot encoded test labels.
    quant_type : str
        One of: 'int8', 'float16', or 'dynamic'.
    filename : str
        Output TFLite filename.
    """

    # Create the TFLite converter from the trained Keras model.
    converter = tf.lite.TFLiteConverter.from_keras_model(model)

    # Step 1: Apply quantization settings.
    if quant_type == 'int8':
        # (a) Enable default optimizations.
        # (b) Provide representative_data_gen(X_train_scaled).
        # (c) Set supported_ops to TFLITE_BUILTINS_INT8.
        # (d) Set inference_input_type and inference_output_type to tf.int8.

        # <-- Enter your code here <--#
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.representative_dataset = lambda: representative_data_gen(X_train_scaled)
        converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        converter.inference_input_type = tf.int8
        converter.inference_output_type = tf.int8
        

    elif quant_type == 'float16':
        # (a) Enable default optimizations.
        # (b) Set supported_types to [tf.float16].

        # <-- Enter your code here <--#
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.target_spec.supported_types = [tf.float16]

    elif quant_type == 'dynamic':
        # (a) Enable default optimizations.

        # <-- Enter your code here <--#
        converter.optimizations = [tf.lite.Optimize.DEFAULT]

    else:
        raise ValueError("quant_type must be one of: 'int8', 'float16', or 'dynamic'.")

    # Step 2: Convert the model and save it to the provided filename.

    # <-- Enter your code here <--#
    tflite_model = converter.convert()
    with open(filename, "wb") as f:
        f.write(tflite_model)

    # Step 3: Run TFLite inference.
    # Complete the following:
    # - Use tf.lite.Interpreter to load the TFLite model.
    # - Allocate tensors.
    # - Get input and output tensor details.
    # - If the input is quantized, quantize each test sample using scale and zero point.
    # - If the output is quantized, dequantize the prediction using scale and zero point.
    # - Collect predictions into y_pred using np.argmax.
    # - Compare with y_true = np.argmax(y_test_cat, axis=1).

    # <-- Enter your code here for TFLite inference <--#
    interpreter = tf.lite.Interpreter(model_path=filename)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    input_scale, input_zero_point = input_details[0]['quantization']
    output_scale, output_zero_point = output_details[0]['quantization']

    y_pred = []
    y_true = np.argmax(y_test_cat, axis=1)

    for i in range(len(X_test)):
        sample = X_test[i:i+1].astype(np.float32)
        if input_details[0]['dtype'] == np.int8:
            sample = (sample / input_scale + input_zero_point).astype(np.int8)
        interpreter.set_tensor(input_details[0]['index'], sample)
        interpreter.invoke()
        output = interpreter.get_tensor(output_details[0]['index'])
        if output_details[0]['dtype'] == np.int8:
            output = (output.astype(np.float32) - output_zero_point) * output_scale
        y_pred.append(np.argmax(output))
    # Step 4: Report results.
    print(f"\n{quant_type.upper()} TFLite model size: {file_size_kb(filename):.2f} KB")

    # <-- Enter your code here: print classification_report and confusion_matrix <--#
    print(classification_report(y_true, y_pred))
    print(confusion_matrix(y_true, y_pred))

In [75]:
# Step 5: Use the function above to create and evaluate three quantized models:
# - 'int8' saved as 'model_int8.tflite'
# - 'float16' saved as 'model_float16.tflite'
# - 'dynamic' saved as 'model_dynamic.tflite'

# <-- Enter your code here <--#
quantize_and_evaluate(model, X_test_scaled, y_test_cat, 'int8', 'model_int8.tflite')
quantize_and_evaluate(model, X_test_scaled, y_test_cat, 'float16', 'model_float16.tflite')
quantize_and_evaluate(model, X_test_scaled, y_test_cat, 'dynamic', 'model_dynamic.tflite')


INFO:tensorflow:Assets written to: /var/folders/1_/r4slzwz964766f02m55dkrzr0000gn/T/tmpf2koaj42/assets


INFO:tensorflow:Assets written to: /var/folders/1_/r4slzwz964766f02m55dkrzr0000gn/T/tmpf2koaj42/assets



INT8 TFLite model size: 5.82 KB
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        19
           1       1.00      1.00      1.00        21
           2       1.00      1.00      1.00        14

    accuracy                           1.00        54
   macro avg       1.00      1.00      1.00        54
weighted avg       1.00      1.00      1.00        54

[[19  0  0]
 [ 0 21  0]
 [ 0  0 14]]


/Users/asafiron-jobes/ai/projects/tinyml-arduino/lib/python3.11/site-packages/tensorflow/lite/python/convert.py:947: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
2026-05-21 11:42:06.317848: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-21 11:42:06.317863: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-21 11:42:06.317969: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/1_/r4slzwz964766f02m55dkrzr0000gn/T/tmpf2koaj42
2026-05-21 11:42:06.318440: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-21 11:42:06.318445: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/1_/r4slzwz964766f02m55dkrzr0000gn/T/tmpf2koaj42
2026-05-21 11:42:06.319735: I tensorflow/cc/saved_model/loader.c

INFO:tensorflow:Assets written to: /var/folders/1_/r4slzwz964766f02m55dkrzr0000gn/T/tmpschqpa55/assets


...

2026-05-21 11:42:06.593485: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-21 11:42:06.593496: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-21 11:42:06.593609: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/1_/r4slzwz964766f02m55dkrzr0000gn/T/tmpschqpa55
2026-05-21 11:42:06.594093: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-21 11:42:06.594097: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/1_/r4slzwz964766f02m55dkrzr0000gn/T/tmpschqpa55
2026-05-21 11:42:06.595429: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-21 11:42:06.616754: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /var/folders/1_/r4slzwz964766f02m55dkrzr0000gn/T/tmpschqpa55
2026-05-

INFO:tensorflow:Assets written to: /var/folders/1_/r4slzwz964766f02m55dkrzr0000gn/T/tmps2p3a2zp/assets


INFO:tensorflow:Assets written to: /var/folders/1_/r4slzwz964766f02m55dkrzr0000gn/T/tmps2p3a2zp/assets



DYNAMIC TFLite model size: 8.24 KB
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        19
           1       1.00      1.00      1.00        21
           2       1.00      1.00      1.00        14

    accuracy                           1.00        54
   macro avg       1.00      1.00      1.00        54
weighted avg       1.00      1.00      1.00        54

[[19  0  0]
 [ 0 21  0]
 [ 0  0 14]]


2026-05-21 11:42:07.001623: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-21 11:42:07.001636: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-21 11:42:07.001742: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/1_/r4slzwz964766f02m55dkrzr0000gn/T/tmps2p3a2zp
2026-05-21 11:42:07.002207: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-21 11:42:07.002212: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/1_/r4slzwz964766f02m55dkrzr0000gn/T/tmps2p3a2zp
2026-05-21 11:42:07.003787: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-21 11:42:07.024857: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /var/folders/1_/r4slzwz964766f02m55dkrzr0000gn/T/tmps2p3a2zp
2026-05-

## Problem 1 - Part (c)

### Pruning

In [76]:
# Step 1: Define a pruning schedule using tfmot.sparsity.keras.PolynomialDecay
# HINT:
# - Use initial_sparsity = 0.5 and final_sparsity = 0.7
# - Set end_step to total training steps (approx. dataset_size / batch_size * epochs)

# <-- Enter your code here <--#
import math

batch_size = 8
epochs_pruning = 10
end_step = math.ceil(len(X_train_scaled) / batch_size) * epochs_pruning

pruning_schedule = tfmot.sparsity.keras.PolynomialDecay(
    initial_sparsity=0.5,
    final_sparsity=0.7,
    begin_step=0,
    end_step=end_step
)

In [77]:
# Step 2: Build a Sequential model with 3 pruned Dense layers:
# - Dense(64, relu)
# - Dense(32, relu)
# - Dense(3, softmax)
# Make sure each Dense layer is wrapped with prune_low_magnitude()

# <-- Enter your code here <--#
prune_low_magnitude = tfmot.sparsity.keras.prune_low_magnitude

model_pruned = Sequential([
    prune_low_magnitude(Dense(64, activation='relu', input_shape=(num_features,)), pruning_schedule=pruning_schedule),
    prune_low_magnitude(Dense(32, activation='relu'), pruning_schedule=pruning_schedule),
    prune_low_magnitude(Dense(num_classes, activation='softmax'), pruning_schedule=pruning_schedule)
])
model_pruned.summary()

Model: "sequential_5"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 prune_low_magnitude_dense_  (None, 64)                1730      
 15 (PruneLowMagnitude)                                          
                                                                 
 prune_low_magnitude_dense_  (None, 32)                4130      
 16 (PruneLowMagnitude)                                          
                                                                 
 prune_low_magnitude_dense_  (None, 3)                 197       
 17 (PruneLowMagnitude)                                          
                                                                 
Total params: 6057 (23.67 KB)
Trainable params: 3075 (12.01 KB)
Non-trainable params: 2982 (11.66 KB)
_________________________________________________________________


In [78]:
# Step 3: Compile the model with categorical_crossentropy and accuracy
# - Train for 10 epochs with batch_size=8 and validation_split=0.2
# - Add tfmot.sparsity.keras.UpdatePruningStep() to the callbacks list

# <-- Enter your code here <--#
model_pruned.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model_pruned.fit(X_train_scaled, y_train_cat, epochs=10, batch_size=8, validation_split=0.2,
                 callbacks=[tfmot.sparsity.keras.UpdatePruningStep()])

Epoch 1/10
13/13 [==============================] - 0s 6ms/step - loss: 1.0195 - accuracy: 0.4444 - val_loss: 0.7561 - val_accuracy: 0.7600
Epoch 2/10
13/13 [==============================] - 0s 2ms/step - loss: 0.6810 - accuracy: 0.8384 - val_loss: 0.5337 - val_accuracy: 0.8400
Epoch 3/10
13/13 [==============================] - 0s 2ms/step - loss: 0.4651 - accuracy: 0.9192 - val_loss: 0.3963 - val_accuracy: 0.9200
Epoch 4/10
13/13 [==============================] - 0s 2ms/step - loss: 0.3290 - accuracy: 0.9596 - val_loss: 0.2997 - val_accuracy: 0.9200
Epoch 5/10
13/13 [==============================] - 0s 2ms/step - loss: 0.2421 - accuracy: 0.9697 - val_loss: 0.2396 - val_accuracy: 0.9200
Epoch 6/10
13/13 [==============================] - 0s 2ms/step - loss: 0.1804 - accuracy: 0.9899 - val_loss: 0.1977 - val_accuracy: 0.9600
Epoch 7/10
13/13 [==============================] - 0s 2ms/step - loss: 0.1438 - accuracy: 0.9899 - val_loss: 0.1700 - val_accuracy: 0.9600
Epoch 8/10
13/13 [==

In [79]:
# Step 4: Remove pruning wrappers using tfmot.sparsity.keras.strip_pruning().
# Then convert the stripped model to TFLite and save it as "model_pruned.tflite".
# Print the final file size in KB.

# Important: converting the unstripped pruned model can keep extra pruning variables
# and make the saved model larger than expected.

# <-- Enter your code here <--#
model_stripped = tfmot.sparsity.keras.strip_pruning(model_pruned)

converter = tf.lite.TFLiteConverter.from_keras_model(model_stripped)
tflite_pruned = converter.convert()
with open("model_pruned.tflite", "wb") as f:
    f.write(tflite_pruned)
print(f"Pruned model size: {file_size_kb('model_pruned.tflite'):.2f} KB")


INFO:tensorflow:Assets written to: /var/folders/1_/r4slzwz964766f02m55dkrzr0000gn/T/tmpmtopkc_u/assets


INFO:tensorflow:Assets written to: /var/folders/1_/r4slzwz964766f02m55dkrzr0000gn/T/tmpmtopkc_u/assets


Pruned model size: 14.18 KB


2026-05-21 11:42:07.958707: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-21 11:42:07.958720: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-21 11:42:07.958823: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/1_/r4slzwz964766f02m55dkrzr0000gn/T/tmpmtopkc_u
2026-05-21 11:42:07.959229: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-21 11:42:07.959233: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/1_/r4slzwz964766f02m55dkrzr0000gn/T/tmpmtopkc_u
2026-05-21 11:42:07.960103: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-21 11:42:07.968157: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /var/folders/1_/r4slzwz964766f02m55dkrzr0000gn/T/tmpmtopkc_u
2026-05-

In [80]:
# Step 5: Evaluate using the stripped model
# - Use np.argmax for predictions
# - Print classification_report and confusion_matrix

# <-- Enter your code here <--#
y_pred_pruned = np.argmax(model_stripped.predict(X_test_scaled), axis=1)
y_true = np.argmax(y_test_cat, axis=1)
print(classification_report(y_true, y_pred_pruned))
print(confusion_matrix(y_true, y_pred_pruned))

2/2 [==============================] - 0s 954us/step
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        19
           1       1.00      1.00      1.00        21
           2       1.00      1.00      1.00        14

    accuracy                           1.00        54
   macro avg       1.00      1.00      1.00        54
weighted avg       1.00      1.00      1.00        54

[[19  0  0]
 [ 0 21  0]
 [ 0  0 14]]


## Problem 1 - Part (d)

### Knowledge Distillation

In [81]:
# Step 1: Define a Sequential model for Student with:
# - Dense(32, relu)
# - Dense(16, relu)
# - Dense(3, softmax)

# <-- Enter your code here <--#
student_model = Sequential([
    Dense(32, activation='relu', input_shape=(num_features,)),
    Dense(16, activation='relu'),
    Dense(num_classes, activation='softmax')
])
student_model.summary()

Model: "sequential_6"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_18 (Dense)            (None, 32)                448       
                                                                 
 dense_19 (Dense)            (None, 16)                528       
                                                                 
 dense_20 (Dense)            (None, 3)                 51        
                                                                 
Total params: 1027 (4.01 KB)
Trainable params: 1027 (4.01 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [82]:
# Step 2: Use model.predict() on X_train_scaled to obtain teacher soft labels

# <-- Enter your code here <--#
teacher_preds_soft = model.predict(X_train_scaled)

4/4 [==============================] - 0s 587us/step


In [83]:
# Step 3:
# (a) Concatenate hard (y_train_cat) and soft (teacher_preds_soft) labels along axis=1
#     to create a combined label for distillation
# (b) Define a custom distillation_loss() function that:
#     - Splits y_true_combined into y_true_hard and y_true_soft
#     - Computes two losses (both using categorical_crossentropy)
#     - Combines them with a weight factor alpha = 0.5

# Hint: Use slicing [:, :3] and [:, 3:] to split the combined labels

# <-- Enter your code here <--#
y_train_combined = np.concatenate([y_train_cat, teacher_preds_soft], axis=1)

def distillation_loss(y_true_combined, y_pred):

    # <-- Enter your code here: implement hard/soft label separation and weighted loss <--#
    y_true_hard = y_true_combined[:, :num_classes]
    y_true_soft = y_true_combined[:, num_classes:]
    alpha = 0.5
    hard_loss = tf.keras.losses.categorical_crossentropy(y_true_hard, y_pred)
    soft_loss = tf.keras.losses.categorical_crossentropy(y_true_soft, y_pred)
    return alpha * hard_loss + (1 - alpha) * soft_loss

In [84]:
# Step 4: Compile the student model with Adam optimizer and distillation_loss
# - Train for 10 epochs, batch_size=8, validation_split=0.2

# <-- Enter your code here <--#
student_model.compile(optimizer='adam', loss=distillation_loss, metrics=['accuracy'])
student_model.fit(X_train_scaled, y_train_combined, epochs=10, batch_size=8, validation_split=0.2)

Epoch 1/10
13/13 [==============================] - 0s 6ms/step - loss: 0.8669 - accuracy: 0.6061 - val_loss: 0.7316 - val_accuracy: 0.8000
Epoch 2/10
13/13 [==============================] - 0s 2ms/step - loss: 0.7209 - accuracy: 0.7778 - val_loss: 0.6205 - val_accuracy: 0.8800
Epoch 3/10
13/13 [==============================] - 0s 2ms/step - loss: 0.5954 - accuracy: 0.9495 - val_loss: 0.5332 - val_accuracy: 0.9200
Epoch 4/10
13/13 [==============================] - 0s 1ms/step - loss: 0.4868 - accuracy: 0.9596 - val_loss: 0.4610 - val_accuracy: 0.9200
Epoch 5/10
13/13 [==============================] - 0s 2ms/step - loss: 0.3992 - accuracy: 0.9596 - val_loss: 0.4045 - val_accuracy: 0.9200
Epoch 6/10
13/13 [==============================] - 0s 1ms/step - loss: 0.3274 - accuracy: 0.9697 - val_loss: 0.3591 - val_accuracy: 0.9200
Epoch 7/10
13/13 [==============================] - 0s 2ms/step - loss: 0.2710 - accuracy: 0.9798 - val_loss: 0.3237 - val_accuracy: 0.9200
Epoch 8/10
13/13 [==

In [85]:
# Step 5: Convert the student model to TFLite.
# - Save it as "model_kd.tflite".
# - Print the file size in KB.

# <-- Enter your code here <--#
converter = tf.lite.TFLiteConverter.from_keras_model(student_model)
tflite_kd = converter.convert()
with open("model_kd.tflite", "wb") as f:
    f.write(tflite_kd)
print(f"KD student model size: {file_size_kb('model_kd.tflite'):.2f} KB")


INFO:tensorflow:Assets written to: /var/folders/1_/r4slzwz964766f02m55dkrzr0000gn/T/tmp1jey6zz7/assets


INFO:tensorflow:Assets written to: /var/folders/1_/r4slzwz964766f02m55dkrzr0000gn/T/tmp1jey6zz7/assets


KD student model size: 6.14 KB


2026-05-21 11:42:08.724341: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-21 11:42:08.724356: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-21 11:42:08.724465: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/1_/r4slzwz964766f02m55dkrzr0000gn/T/tmp1jey6zz7
2026-05-21 11:42:08.724975: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-21 11:42:08.724979: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/1_/r4slzwz964766f02m55dkrzr0000gn/T/tmp1jey6zz7
2026-05-21 11:42:08.726223: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-21 11:42:08.745854: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /var/folders/1_/r4slzwz964766f02m55dkrzr0000gn/T/tmp1jey6zz7
2026-05-

In [86]:
# Step 6: Use student_model.predict() to obtain predictions on X_test_scaled
# - Print classification_report and confusion_matrix

# <-- Enter your code here <--#
y_pred_kd = np.argmax(student_model.predict(X_test_scaled), axis=1)
y_true = np.argmax(y_test_cat, axis=1)
print(classification_report(y_true, y_pred_kd))
print(confusion_matrix(y_true, y_pred_kd))

2/2 [==============================] - 0s 908us/step
              precision    recall  f1-score   support

           0       0.95      0.95      0.95        19
           1       0.95      0.95      0.95        21
           2       1.00      1.00      1.00        14

    accuracy                           0.96        54
   macro avg       0.97      0.97      0.97        54
weighted avg       0.96      0.96      0.96        54

[[18  1  0]
 [ 1 20  0]
 [ 0  0 14]]


## Problem 1 - Part (e)

### Possibility of Further Model Size Reduction

Can you **further reduce the model size** beyond the smallest model obtained in parts **(b)**, **(c)**, or **(d)**, **without sacrificing significant classification performance**?

Your task is to:

1. **Analyze and compare** the results from previous parts: Which model had the smallest size? Which performed best?

2. **Propose a strategy** that combines or enhances techniques learned so far.

3. **Implement** your proposed solution.

4. **Evaluate** the resulting model using both:
   - TFLite model size (in KB)
   - Classification performance (accuracy and report)

5. **Justify your results:**
   - If further size reduction is **not** possible without major loss of accuracy, explain why.
   - If you succeed in reducing the size **further**, highlight what change made the biggest difference.


### **Note:** If this part includes any code, please include it below. The related discussion should be submitted as part of your PDF that contains answers to all [Dis] questions in this assignment.


In [87]:
# <-- (if needed) Enter your code here <--#
quantize_and_evaluate(student_model, X_test_scaled, y_test_cat, 'int8', 'model_kd_int8.tflite')

INFO:tensorflow:Assets written to: /var/folders/1_/r4slzwz964766f02m55dkrzr0000gn/T/tmpi6l5bz8k/assets


INFO:tensorflow:Assets written to: /var/folders/1_/r4slzwz964766f02m55dkrzr0000gn/T/tmpi6l5bz8k/assets



INT8 TFLite model size: 3.68 KB
              precision    recall  f1-score   support

           0       0.95      0.95      0.95        19
           1       0.95      0.95      0.95        21
           2       1.00      1.00      1.00        14

    accuracy                           0.96        54
   macro avg       0.97      0.97      0.97        54
weighted avg       0.96      0.96      0.96        54

[[18  1  0]
 [ 1 20  0]
 [ 0  0 14]]


/Users/asafiron-jobes/ai/projects/tinyml-arduino/lib/python3.11/site-packages/tensorflow/lite/python/convert.py:947: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
2026-05-21 11:42:09.040290: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-21 11:42:09.040302: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-21 11:42:09.040421: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/1_/r4slzwz964766f02m55dkrzr0000gn/T/tmpi6l5bz8k
2026-05-21 11:42:09.040958: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-21 11:42:09.040962: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/1_/r4slzwz964766f02m55dkrzr0000gn/T/tmpi6l5bz8k
2026-05-21 11:42:09.042264: I tensorflow/cc/saved_model/loader.c

# Problem 2: Exploring Edge Impulse (20 points)


### Note

Problem 2 consists entirely of discussion questions. Submit your responses in the same PDF file that contains answers to the other **[Dis]** questions in this assignment.

Before submission, make sure this notebook runs with the **Python (tinyml-arduino)** kernel and that all requested outputs are visible. Host this notebook and your discussion PDF in your public GitHub repository, then submit the repository link through Canvas.
